# 🛢️ Semana 4 · Unidad 2 — Laboratorio: El Problema del Derrame de Petróleo

---

> **Curso:** Estructuras de Datos y Algoritmos  
> **Unidad:** 2 — Tipos de Datos Abstractos  
> **Entrega:** Sube este archivo como `A2_<apellido>_<nombre>.ipynb`  
> ⚠️ El archivo `unionfind.py` debe estar en la misma carpeta que este notebook.

---

## 🎯 Objetivo

En este assignment **no implementarás** Union-Find desde cero. Usarás la librería `unionfind.py` entregada por el curso — exactamente como se hace en la industria: conoces la **API** y la usas sin ver la implementación interna.

Esto es el núcleo del **Tipo de Dato Abstracto (TDA)**:

```
┌──────────────────────────────────────────────────────┐
│              Tu código  →  clase OilSpill            │
│        usa la API sin conocer la implementación      │
└──────────────────────────┬───────────────────────────┘
                           │  from unionfind import ...
┌──────────────────────────▼───────────────────────────┐
│           unionfind.py   (caja negra 🔒)             │
│      WeightedQuickUnion — detalles ocultos           │
└──────────────────────────────────────────────────────┘
```

---

## 📖 El problema

Modelamos el océano como una **grilla N×N**. Cada celda es `True` (contaminada) o `False` (limpia).

```
     col→  0   1   2   3   4
fila ↓  ┌───┬───┬───┬───┬───┐
  0     │ . │ . │ █ │ . │ . │
        ├───┼───┼───┼───┼───┤
  1     │ . │ █ │ █ │ . │ . │
        ├───┼───┼───┼───┼───┤
  2     │ . │ █ │ . │ . │ █ │
        ├───┼───┼───┼───┼───┤
  3     │ . │ █ │ █ │ █ │ █ │
        ├───┼───┼───┼───┼───┤
  4     │ . │ . │ █ │ . │ . │
        └───┴───┴───┴───┴───┘
  █ = contaminada    . = limpia
```

Dos celdas son parte de la **misma mancha** si están conectadas horizontal o verticalmente (no diagonal).

Tu clase `OilSpill` debe responder:
1. ¿Cuántas manchas independientes hay?
2. ¿Cuántas celdas tiene la mancha más grande?
3. ¿Dos celdas están en la misma mancha?
4. ¿Una mancha toca algún borde de la grilla?

---

## 📚 API de la librería `unionfind`

Esta sección es tu **referencia completa**. No necesitas leer el código fuente de `unionfind.py`.

---

### Importación

```python
from unionfind import WeightedQuickUnion
```

---

### Constructor

```python
uf = WeightedQuickUnion(n)
```
Crea una estructura con `n` objetos identificados `0, 1, ..., n-1`.  
Al inicio hay `n` componentes (cada objeto es su propio componente).

---

### Métodos disponibles

| Método | Qué hace | Retorna | Costo |
|--------|----------|---------|-------|
| `uf.union(p, q)` | Conecta el componente de `p` con el de `q`. Si ya están conectados, no hace nada. | `None` | O(log N) |
| `uf.find(p)` | Identificador del componente al que pertenece `p`. | `int` | O(log N) |
| `uf.connected(p, q)` | `True` si `p` y `q` están en el mismo componente. | `bool` | O(log N) |
| `uf.count()` | Número de componentes actuales. | `int` | O(1) |
| `uf.component_size(p)` | Número de objetos en el mismo componente que `p`. | `int` | O(log N) |

---

### Ejemplo de uso

```python
from unionfind import WeightedQuickUnion

uf = WeightedQuickUnion(5)    # {0} {1} {2} {3} {4}  → 5 componentes

uf.union(0, 1)                # {0,1} {2} {3} {4}    → 4 componentes
uf.union(1, 2)                # {0,1,2} {3} {4}       → 3 componentes

uf.connected(0, 2)            # True  — mismo componente
uf.connected(0, 3)            # False — componentes distintos

uf.count()                    # 3
uf.component_size(0)          # 3  (componente {0,1,2})
uf.component_size(3)          # 1  (componente {3})

uf.find(0) == uf.find(2)      # True  — mismo id de componente
uf.find(0) == uf.find(3)      # False — distinto id de componente
```

---

> ⚠️ Los índices deben estar entre `0` y `n-1`. Un índice fuera de rango lanza `IndexError`.

---
## ⚙️ Celda 1 — Setup *(ejecutar primero, no modificar)*

In [ ]:
# ════════════════════════════════════════════════════════════
#  SETUP — NO MODIFICAR
# ════════════════════════════════════════════════════════════
from unionfind import WeightedQuickUnion
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def visualizar_grilla(grid, titulo="Grilla", manchas=None):
    n = len(grid)
    fig, ax = plt.subplots(figsize=(min(10, n*0.9+1), min(10, n*0.9+1)))
    if manchas:
        ids = list(set(manchas.values()))
        cmap = plt.cm.tab20(np.linspace(0, 1, max(len(ids), 2)))
        cmap_d = {mid: cmap[i] for i, mid in enumerate(ids)}
    for r in range(n):
        for c in range(n):
            if grid[r][c]:
                col = cmap_d[manchas[(r,c)]] if manchas and (r,c) in manchas else '#2d3436'
                rect = plt.Rectangle([c,n-1-r],1,1,facecolor=col,edgecolor='white',lw=0.5)
            else:
                rect = plt.Rectangle([c,n-1-r],1,1,facecolor='#dfe6e9',edgecolor='white',lw=0.5)
            ax.add_patch(rect)
    ax.set_xlim(0,n); ax.set_ylim(0,n)
    ax.set_xticks(np.arange(n)+0.5); ax.set_xticklabels(range(n))
    ax.set_yticks(np.arange(n)+0.5); ax.set_yticklabels(range(n-1,-1,-1))
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.set_xlabel('columna'); ax.set_ylabel('fila')
    plt.tight_layout(); plt.show()

# Verificación rápida de la librería
_uf = WeightedQuickUnion(5)
_uf.union(0,1); _uf.union(1,2)
assert _uf.connected(0,2) and not _uf.connected(0,3) and _uf.count()==3
print("✅ unionfind.py cargada y verificada")
print(f"   {_uf}")

---
## 📐 Celda 2 — Mapeo grilla 2D → índice 1D

`WeightedQuickUnion` trabaja con enteros `0..N²-1`. Necesitas convertir cada celda:

```
índice = fila × N + col

N=5:  (0,0)→0  (0,1)→1  ...  (0,4)→4
      (1,0)→5  (1,1)→6  ...  (1,4)→9
      ...
      (4,4)→24
```

Los 4 vecinos de `(r,c)` son `(r-1,c)`, `(r+1,c)`, `(r,c-1)`, `(r,c+1)`.  
Solo se conectan vecinos **dentro de la grilla** y **contaminados**.

---
## 💻 Celda 3 — Tu implementación

In [ ]:
class OilSpill:
    """
    Modela un derrame de petróleo en una grilla N×N.
    Usa WeightedQuickUnion (de la librería unionfind) internamente.

    Parámetro
    ---------
    grid : list[list[bool]]
        grid[r][c] == True  →  celda (r,c) contaminada
        grid[r][c] == False →  celda (r,c) limpia
    """

    def __init__(self, grid: list):
        self.grid = grid
        self.n    = len(grid)

        # TODO 1: Crea un WeightedQuickUnion con N*N elementos
        self.uf = # TODO

        # TODO 2: Recorre todas las celdas contaminadas y únelas
        #         con sus vecinos contaminados usando self.uf.union()
        #         Usa self._idx(r, c) para obtener el índice.


    def _idx(self, r: int, c: int) -> int:
        """Convierte (fila, col) al índice del arreglo 1D."""
        return r * self.n + c


    def num_manchas(self) -> int:
        """
        TODO 3: Número de manchas (componentes de celdas contaminadas).
        Pista: set de uf.find() sobre celdas contaminadas.
        """
        pass

    def mancha_mas_grande(self) -> int:
        """
        TODO 4: Tamaño en celdas de la mancha más grande. 0 si no hay.
        Pista: usa uf.component_size() sobre celdas contaminadas.
        """
        pass

    def misma_mancha(self, r1:int, c1:int, r2:int, c2:int) -> bool:
        """
        TODO 5: True si (r1,c1) y (r2,c2) están en la misma mancha.
        False si alguna celda está limpia.
        Pista: usa uf.connected().
        """
        pass

    def llega_al_borde(self, r:int, c:int) -> bool:
        """
        TODO 6: True si la mancha de (r,c) toca algún borde.
        False si la celda está limpia.
        Pista: itera las celdas del borde y usa uf.connected().
        """
        pass

    def mapa_manchas(self) -> dict:
        """
        TODO 7: Retorna { (r,c): id_mancha } para celdas contaminadas.
        id_mancha = uf.find() de esa celda. Usado para visualizar.
        """
        pass


print("Clase OilSpill definida.")

---
## 🔍 Celda 4 — Exploración

In [ ]:
grid_ejemplo = [
    [False, False, True,  False, False],
    [False, True,  True,  False, False],
    [False, True,  False, False, True ],
    [False, True,  True,  True,  True ],
    [False, False, True,  False, False],
]
oil = OilSpill(grid_ejemplo)
print(f"Manchas            : {oil.num_manchas()}")
print(f"Mancha más grande  : {oil.mancha_mas_grande()} celdas")
print(f"¿(0,2)-(4,2) juntas? {oil.misma_mancha(0,2,4,2)}")
print(f"¿(2,4)-(3,3) juntas? {oil.misma_mancha(2,4,3,3)}")
print(f"¿(0,2) al borde?     {oil.llega_al_borde(0,2)}")
print(f"¿(2,4) al borde?     {oil.llega_al_borde(2,4)}")
visualizar_grilla(grid_ejemplo, titulo="Derrame — manchas por componente", manchas=oil.mapa_manchas())

In [ ]:
# Crea tu propia grilla y experimenta
mi_grid = [
    [True,  False, False, True ],
    [True,  False, False, True ],
    [False, False, True,  True ],
    [False, True,  True,  False],
]
mi_oil = OilSpill(mi_grid)
print(f"Manchas: {mi_oil.num_manchas()}  |  Más grande: {mi_oil.mancha_mas_grande()} celdas")
visualizar_grilla(mi_grid, titulo="Mi grilla", manchas=mi_oil.mapa_manchas())

---
## 🧪 Celda 5 — Tests automáticos *(no modificar)*

In [ ]:
def ejecutar_tests():
    R = []
    def caso(n, c, d=""): R.append((n, bool(c), str(d)))

    g = [[False,False,True,False,False],[False,True,True,False,False],
         [False,True,False,False,True],[False,True,True,True,True],[False,False,True,False,False]]
    o = OilSpill(g)

    caso("T1  num_manchas",             o.num_manchas()==2,              f"esp=2 obt={o.num_manchas()}")
    caso("T2  mancha_mas_grande",        o.mancha_mas_grande()==9,        f"esp=9 obt={o.mancha_mas_grande()}")
    caso("T3  misma_mancha True",        o.misma_mancha(0,2,4,2),         "(0,2)-(4,2) misma mancha")
    caso("T4  misma_mancha False",       not o.misma_mancha(0,2,2,4),     "(0,2)-(2,4) distintas")
    caso("T5  misma_mancha limpia",      not o.misma_mancha(0,0,0,2),     "(0,0) limpia")
    caso("T6  llega_al_borde True",      o.llega_al_borde(0,2),           "toca fila 0")
    caso("T7  llega_al_borde False",     not o.llega_al_borde(2,4),       "(2,4) no toca borde")
    mapa = o.mapa_manchas()
    cont = [(r,c) for r in range(5) for c in range(5) if g[r][c]]
    limp = [(r,c) for r in range(5) for c in range(5) if not g[r][c]]
    caso("T8  mapa completo",            all(x in mapa for x in cont),    "faltan celdas")
    caso("T9  mapa sin limpias",         all(x not in mapa for x in limp),"hay limpias")
    gv=[[False]*3 for _ in range(3)]; ov=OilSpill(gv)
    caso("T10 grilla vacía",             ov.num_manchas()==0 and ov.mancha_mas_grande()==0, "")
    gf=[[True]*3 for _ in range(3)]; of=OilSpill(gf)
    caso("T11 3×3 llena",                of.num_manchas()==1 and of.mancha_mas_grande()==9,"")
    ga=[[True,False,True],[False,False,False],[True,False,True]]; oa=OilSpill(ga)
    caso("T12 4 aisladas",               oa.num_manchas()==4,             f"obt={oa.num_manchas()}")
    caso("T13 simetría",                 o.misma_mancha(0,2,4,2)==o.misma_mancha(4,2,0,2),"")

    print("="*50)
    ok=0
    for nombre,passed,det in R:
        print(f"  {'✅ PASS' if passed else '❌ FAIL'}  {nombre}")
        if not passed and det: print(f"         → {det}")
        if passed: ok+=1
    print("="*50)
    print(f"  {ok}/{len(R)} tests pasados")
    print("="*50)

try:
    ejecutar_tests()
except Exception as e:
    print(f"⚠️  {e}")

---
## ✍️ Celda 6 — Preguntas de reflexión

**Pregunta 1:** Usaste `WeightedQuickUnion` sin ver su código. ¿Qué ventaja tiene ese enfoque? ¿Qué pasaría si la cátedra cambiara la implementación interna pero mantuviera la misma API?

> *Tu respuesta aquí...*

**Pregunta 2:** `num_manchas()` no puede usar `self.uf.count()` directamente. ¿Por qué? ¿Qué contaría de más?

> *Tu respuesta aquí...*

**Pregunta 3:** ¿Cuál es la complejidad de `__init__` en función de N? Considera el recorrido y el costo de cada `union()`. Justifica.

> *Tu respuesta aquí...*

---
## ✅ Celda 7 — Checklist de entrega

- [ ] `Kernel → Restart & Run All` sin errores
- [ ] Tests muestran puntaje final
- [ ] Las 3 preguntas respondidas
- [ ] Archivo nombrado `A2_<apellido>_<nombre>.ipynb`
- [ ] `unionfind.py` en la misma carpeta al momento de ejecutar

**Nombre:** ___________________________  
**Fecha:** ___________________________